In [ ]:
import numpy as np
import joblib
from utils import train_categorical_model, prepare_datasets, validate_results, train_incident_agent, generate_oof_features
from rcf_model import RCF
from meta_scorer import train_fusion_meta_learner, CostSensitiveMetaLearner, _incident_entropy
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
file_path = r"UNSW-NB15\unsw_train.csv"

# FIX (PCA Leakage): prepare_datasets now returns X_num_raw (6th value) —
# the raw, unscaled numerical frame. This is passed to generate_oof_features
# so each fold fits its own scaler+PCA exclusively on its training split.
# X_train_num (PCA-transformed using the full-train artifact) is kept for
# the final RCF training step in cell 5.
X_train_cat, X_train_num, X_train_num_raw, y_bin, cat_cols, y_multi = prepare_datasets(file_path, is_train=True)

# FIX (PCA Leakage): Pass X_train_num_raw instead of X_train_num.
# generate_oof_features will fit a fresh scaler+PCA per fold internally.
oof_cat_scores, oof_rcf_scores, oof_incident_entropy = generate_oof_features(
    X_train_cat, X_train_num_raw, y_bin, cat_cols,
    train_cat_func=train_categorical_model,
    rcf_class=RCF,
    train_incident_func=train_incident_agent,
    n_splits=5,
    y_multi=y_multi
)

In [ ]:
X_meta_train_clean = np.column_stack((oof_cat_scores, oof_rcf_scores, oof_incident_entropy))

meta_learner = train_fusion_meta_learner(
    X_meta_train_clean,
    y_bin,
    COST_FN=10,
    COST_FP=2,
    lambda_reg=0.1
)

meta_learner.save_model("Saves/meta_learner.pkl")

In [ ]:
# Train full-dataset base models (no OOF needed here — these are frozen for inference)
catboost_model = train_categorical_model(X_train_cat, y_bin, cat_cols)
incident_model = train_incident_agent(X_train_cat, y_multi, cat_cols)

print("\nSaving Base Models to disk...")
catboost_model.save_model(r"Saves/catboost_base.cbm")
incident_model.save_model(r"Saves/incident_base.cbm")

In [ ]:
# X_train_num is the full-train PCA output from prepare_datasets — correct input
# for the final RCF. X_train_num_raw is NOT used here because the saved PCA
# artifact must be consistent with what predict_proba receives at test time.
rcf_model = RCF(num_trees=40, tree_size=256)
y_bin_reset = y_bin.reset_index(drop=True)
X_train_num_normal_full = X_train_num[y_bin_reset == 0]
rcf_model.fit_predict(X_train_num_normal_full)

rcf_model.save_model(r"Saves/rcf_base.pkl")

In [ ]:
print("\n=======================================================")
print("FINAL HOLD-OUT TEST SET EVALUATION")
print("=======================================================\n")

test_file_path = r"UNSW-NB15\unsw_test.csv"

# Test mode: prepare_datasets returns X_num already transformed by the saved
# full-train scaler+PCA. X_num_raw is unused in the test path but unpacked
# for API consistency.
X_test_cat, X_test_num, _, y_test_bin, cat_cols_test, y_test_multi = prepare_datasets(test_file_path, is_train=False)

# Verify PCA dimensionality matches the saved artifact (guards against
# retraining with a different variance threshold or different data)
saved_pca = joblib.load(r"Saves/feature_pca.pkl")
expected_pca_dims = saved_pca.n_components_
assert X_test_num.shape[1] == expected_pca_dims, (
    f"PCA dimensionality mismatch: expected {expected_pca_dims} components, "
    f"got {X_test_num.shape[1]}. Retrain or reload the correct scaler/PCA artifacts."
)

# Load frozen base models
print("\nLoading frozen base models from disk...")
loaded_catboost = CatBoostClassifier()
loaded_catboost.load_model("Saves/catboost_base.cbm")
loaded_rcf = RCF.load_model("Saves/rcf_base.pkl")
loaded_incident = CatBoostClassifier()
loaded_incident.load_model("Saves/incident_base.cbm")

# Generate Level-1 base scores
print("Generating base model scores (CatBoost, RCF, Incident)...")
cat_scores_test = loaded_catboost.predict_proba(X_test_cat)[:, 1]
rcf_scores_test_norm = loaded_rcf.predict_proba(X_test_num)

incident_proba_test = loaded_incident.predict_proba(X_test_cat)
incident_entropy_test = _incident_entropy(incident_proba_test)

# Fuse through meta-learner
print("Fusing scores through the Cost-Sensitive Meta-Learner...")
X_meta_test = np.column_stack((cat_scores_test, rcf_scores_test_norm, incident_entropy_test))
loaded_meta_learner = CostSensitiveMetaLearner.load_model("Saves/meta_learner.pkl")
test_final_risk = loaded_meta_learner.predict_proba(X_meta_test)

# Zero Trust boundary
test_predictions = (test_final_risk > 0.5).astype(int)

# SOC operational metrics
tn, fp, fn, tp = confusion_matrix(y_test_bin, test_predictions).ravel()
final_soc_cost = (fn * 10) + (fp * 2)

print("\n--- FINAL OPERATIONAL REPORT (TEST SET) ---")
print(classification_report(y_test_bin, test_predictions, target_names=["Normal (0)", "Attack (1)"]))

print("\n--- ZERO TRUST SOC IMPACT ---")
print(f"True Negatives (Traffic Safely Allowed): {tn}")
print(f"True Positives (Attacks Successfully Blocked): {tp}")
print(f"False Positives (Unjustified Alert Fatigue): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")
print("-------------------------------------------------------")
print(f"Total Operational Penalty Score: {final_soc_cost}")